# MVP - Banco de Dados para Análise de Risco Operacional em Consórcios Imobiliários

**Nome:** _Paulo Roberto Chiquito_  
**Matrícula:** _4052026000805_  
**Data:** _27/09/2026_

**Ambiente:** Databricks Free Edition



## Objetivo

Construir uma solução mínima viável de dados para organizar, tratar, armazenar e analisar informações públicas sobre grupos de consórcios imobiliários, com foco em transformar os dados operacionais em indicadores comparáveis que permitam:
- Avaliar a concentração de grupos e cotas entre as administradoras.
- Identificar administradoras com maior exposição a potenciais riscos operacionais.
- Detectar grupos com níveis elevados de inadimplência.
- Comparar o desempenho relativo das administradoras por meio de taxas e proporções.
- Investigar possíveis fatores associados à inadimplência, como prazo dos grupos e taxa de administração.
- Validar a consistência dos cálculos realizados a partir dos dados detalhados em relação aos dados consolidados disponibilizados




## Etapa 2 e 4.1 — Contexto de Negócios e Perguntas

### Contexto de Negócio

O sistema de consórcios imobiliários é composto por diversas administradoras que operam centenas de grupos com características bastante distintas, incluindo quantidade de cotas, prazo de duração, valor dos créditos, taxas de administração e estágio de maturação dos grupos. Essa heterogeneidade dificulta comparações diretas baseadas apenas em números absolutos, uma vez que administradoras de maior porte tendem naturalmente a concentrar mais participantes, maior volume financeiro e maiores quantidades de ocorrências operacionais.

Para permitir uma avaliação mais justa e comparável entre administradoras e grupos de diferentes tamanhos, torna-se necessário utilizar indicadores relativos, capazes de medir a qualidade e a saúde operacional de cada operação independentemente de sua escala.

Nesse contexto, a inadimplência é um dos principais indicadores de atenção, pois afeta diretamente a capacidade financeira dos grupos, o fluxo de arrecadação e a sustentabilidade do sistema de consórcios. Entretanto, a análise isolada da inadimplência pode não capturar toda a complexidade operacional dos grupos. Por isso, este estudo também considera indicadores complementares, como a quantidade de cotas excluídas e o número de cotas contempladas com crédito ainda não utilizado, que podem revelar potenciais fragilidades na gestão ou características específicas do comportamento dos consorciados.

A partir dos dados disponibilizados na posição de julho de 2026, serão construídos indicadores proporcionais que permitam comparar administradoras e grupos de forma padronizada, identificando concentrações de risco, padrões operacionais relevantes e possíveis desvios de comportamento. Os resultados poderão auxiliar órgãos reguladores, gestores e analistas na identificação de grupos que demandam acompanhamento mais próximo, além de fornecer uma visão consolidada da situação do mercado de consórcios imobiliários analisado.

### Problema

Diante da diversidade de administradoras e grupos de consórcios imobiliários existentes na base analisada, surge a necessidade de identificar quais operações apresentam maiores sinais de atenção operacional e financeira.

A questão central deste estudo é:

Quais administradoras e grupos de consórcios imobiliários apresentam os maiores sinais de atenção operacional na data-base de julho de 2026, considerando indicadores de inadimplência, exclusão de cotas e utilização dos créditos contemplados?

### Perguntas de Negócio

1. Quais administradoras possuem mais grupos imobiliários?
2. Quais administradoras concentram mais cotas ativas?
3. Quais grupos apresentam maior taxa de inadimplência?
4. Qual é a taxa de inadimplência agregada por administradora?
5. A taxa de inadimplência varia entre faixas de prazo?
6. Existe associação entre taxa de administração e inadimplência?
7. Quais administradoras possuem mais cotas excluídas?
8. Quais administradoras possuem mais cotas com crédito pendente de utilização?
9. Quais grupos combinam alta inadimplência e grande quantidade de cotas excluídas?
10. Os valores calculados a partir dos grupos são compatíveis com o arquivo consolidado?

### Dados Brutos

Os dados utilizados neste trabalho são públicos e provenientes do **Banco Central do Brasil (BCB)**, disponibilizados na página de Estabilidade Financeira — Base de Dados de Consórcios (https://www.bcb.gov.br/estabilidadefinanceira/consorciobd).

**Estrutura dos arquivos utilizados:**

| Arquivo | Descrição | Formato |
|---|---|---|
| `202607Segmentos_Consolidados.csv` | Dados consolidados por segmento de consórcio (não utilizado neste MVP) | CSV (sep. `;`, decimal `,`) |
| `202607Bens_Imoveis_Grupos.csv` | Dados por grupo de consórcio imobiliário — arquivo principal | CSV (sep. `;`, decimal `,`) |
| `Significado_dos_campos_e_metricas.xlsx` | Dicionário de dados com a descrição de cada campo e métrica | Excel (XLSX) |

**Características do arquivo principal (`202607Bens_Imoveis_Grupos.csv`):**
- Separador de campos: ponto e vírgula (`;`)
- Separador decimal: vírgula (`,`)
- Nomes de colunas com acentuação
- CNPJ com zeros à esquerda (texto)
- Data-base no formato `AAAAMM`
- Códigos de grupo alfanuméricos
- Encoding: `windows-1252`

**Granularidade:** Uma linha por administradora, data-base, código do grupo e código do segmento.

### Licença dos Dados

Os dados publicados pelo Banco Central do Brasil na Base de Dados de Consórcios são de **uso público e gratuito**, podendo ser utilizados para fins acadêmicos, de pesquisa e análise. Não foi identificada uma licença formal específica (como Creative Commons) no portal do BCB, mas a política institucional de dados abertos do BCB permite a livre utilização dos dados publicados, desde que citada a fonte.

> **Fonte:** Banco Central do Brasil — Base de Dados de Consórcios — https://www.bcb.gov.br/estabilidadefinanceira/consorciobd — Data-base: julho de 2026.

## Etapa 4.2 — Carga dos Dados

A carga de dados foi realizada inteiramente em um notebook Python no Databricks, utilizando Spark DataFrame API para leitura dos arquivos CSV armazenados em um Unity Catalog Volume (`workspace.default.arquivos_mvp`). O arquivo XLSX (dicionário de dados) foi mantido como metadado de referência, sem necessidade de carga como tabela operacional.

O processo de carga seguiu os seguintes passos:
1. **Configuração dos caminhos** dos arquivos no UC Volume
2. **Verificação da existência** dos arquivos no sistema de arquivos
3. **Criação do schema** `workspace.mvp_consorcios` no Unity Catalog
4. **Leitura do CSV** com preservação de todas as colunas como texto (para não perder zeros à esquerda do CNPJ e controlar conversão decimal manualmente)

> **Script de referência:** O notebook completo está disponível no GitHub: https://github.com/prchiquito83/mvp-consorcios-imobiliarios

### Configuração dos Caminhos dos Arquivos (em Python)

In [0]:
#Configuração dos caminhos dos arquivos - Alterar as informações em catalogo, esquema e volume para o seu ambiente
catalogo = "workspace"
esquema = "default"
volume = "arquivos_mvp"

# Montagem do caminho-base.
caminho_base = f"/Volumes/{catalogo}/{esquema}/{volume}"
arquivo_segmentos = f"{caminho_base}/202607Segmentos_Consolidados.csv"
arquivo_grupos = f"{caminho_base}/202607Bens_Imoveis_Grupos.csv"
arquivo_dicionario = f"{caminho_base}/Significado_dos_campos_e_metricas.xlsx"

print("Arquivo de segmentos:", arquivo_segmentos)
print("Arquivo de grupos:", arquivo_grupos)
print("Dicionário:", arquivo_dicionario)

Arquivo de segmentos: /Volumes/workspace/default/arquivos_mvp/202607Segmentos_Consolidados.csv
Arquivo de grupos: /Volumes/workspace/default/arquivos_mvp/202607Bens_Imoveis_Grupos.csv
Dicionário: /Volumes/workspace/default/arquivos_mvp/Significado_dos_campos_e_metricas.xlsx


### Verficação da Existência dos Arquivos (em Python)

In [0]:
#Verificação da existência dos arquivos nos locais especificados anteriormente (se aparecer NÃO ENCONTRADO, volte ao volume e confira nome do catálogo, nome do schema, nome do volume, grafia do arquivo ou as extensões de arquivos)
import os
arquivos_esperados = [arquivo_segmentos,arquivo_grupos,arquivo_dicionario]

for arquivo in arquivos_esperados:
    existe = os.path.exists(arquivo)
    print(f"{arquivo}: {'ENCONTRADO' if existe else 'NÃO ENCONTRADO'}")

/Volumes/workspace/default/arquivos_mvp/202607Segmentos_Consolidados.csv: ENCONTRADO
/Volumes/workspace/default/arquivos_mvp/202607Bens_Imoveis_Grupos.csv: ENCONTRADO
/Volumes/workspace/default/arquivos_mvp/Significado_dos_campos_e_metricas.xlsx: ENCONTRADO


###Criação do Banco Lógico

In [0]:
%sql
--Crie o schema mvp_consorcios dentro do catálogo workspace, mas somente se ele ainda não existir. (para não gerar erro)
CREATE SCHEMA IF NOT EXISTS workspace.mvp_consorcios;

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA mvp_consorcios;

###Ingestão do CSV de grupos (em Python)

O arquivo CVS utiliza:
* Separador';'
* Vírgula como separador decimal
* Nomes de campos com acentos
* Espaços adicionais em nomes de administradoras
* CNPJ com zeros à esquerda
* data-base no formato AAAAMM


Será realizado a leitura das colunas como texto utilizando a função Spark via python e uma amostra de linhas será gerada.

In [0]:
from pyspark.sql import functions as F

# Leitura inicial mantendo todas as colunas como texto.
df_grupos_bruto = (
spark.read
.option("header", True)
.option("sep", ";")
.option("encoding", "windows-1252")
.option("inferSchema", False)
.csv(arquivo_grupos)
)

print("Quantidade de linhas:", df_grupos_bruto.count())
print("Quantidade de colunas:", len(df_grupos_bruto.columns))

display(df_grupos_bruto.limit(10))

Quantidade de linhas: 2835
Quantidade de colunas: 17


#Nome_da_Administradora,CNPJ_da_Administradora,Data_base,Código_do_grupo,Código_do_segmento,Número_da_assembléia_geral_ordinária,Valor_médio_do_bem,Índice_de_correção,Taxa_de_administração,Prazo_do_grupo_em_meses,Quantidade_de_cotas_ativas_em_dia,Quantidade_de_cotas_ativas_contempladas_inadimplentes,Quantidade_de_cotas_ativas_não_contempladas_inadimplentes,Quantidade_de_cotas_ativas_contempladas_no_mês,Quantidade_de_cotas_excluídas,Quantidade_de_cotas_ativas_quitadas,Quantidade_de_cotas_ativas_com_crédito_pendente_de_utilização
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00057,1,194,"248144,39",3,"20,62128",200,699,48,0,0,2975,459,52
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00058,1,191,"566907,83",3,"18,09440",200,658,39,0,0,2130,346,83
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00061,1,190,"212069,65",3,"21,28078",200,749,51,0,0,2621,460,50
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00062,1,189,"297828,75",3,"18,02103",200,763,30,1,8,2268,371,149
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00063,1,189,"252281,48",3,"10,91300",200,583,17,0,0,655,471,6
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00064,1,189,"219680,78",3,"21,02703",200,769,30,0,0,2891,456,36
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00065,1,188,"252232,75",3,"10,80283",200,570,30,0,0,769,435,15
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00066,1,187,"228547,51",3,"20,12428",200,752,40,3,0,2853,403,76
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00067,1,184,"297777,39",3,"19,34876",200,647,35,1,4,2205,330,71
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00068,1,186,"203130,41",3,"21,39913",200,755,44,1,0,3362,448,67


A leitura inicial como texto foi intencional. Se o Spark fosse feito automaticamente, o CNPJ poderi virar número e perder os zeros existentes à esquerda, os valores com vírgula decimal poderiam vir de maneira incorreta e os códigos alfanuméricos de grupos poderiam gerar inconsistências.

## Etapa 4.4 — Pipeline de Dados

O pipeline ETL foi organizado em um **único notebook** (`MVP_Consorcios_Imobiliarios`), seguindo a arquitetura de camadas **Bronze → Silver → Gold**. A escolha por um único notebook se deu por se tratar de um MVP com uma única data-base (julho de 2026), sem necessidade de orquestração complexa ou cargas incrementais.

**Fluxo do pipeline (ETL):**

1. **Extract** — Leitura dos arquivos CSV do UC Volume, preservando todas as colunas como texto
2. **Transform** — Padronização de nomes de colunas, limpeza de espaços, conversão de tipos (decimal, data, inteiro), criação de indicadores (taxa de inadimplência, cotas ativas calculadas)
3. **Load** — Persistência em três camadas:
   - **Bronze**: dados com colunas normalizadas, próximos ao formato original
   - **Silver**: dados tratados e convertidos para tipos adequados
   - **Gold**: indicadores agregados por administradora e por grupo, com classificações de risco e prazo

**Estrutura do notebook (sequência de etapas):**

| Seção | Células | Descrição |
|---|---|---|
| Contexto e Perguntas | 1-2 | Objetivo, contexto, perguntas, dados brutos e licença |
| Carga dos Dados | 3-12 | Configuração, verificação e leitura do CSV |
| Pipeline de Dados | 13 | Visão geral do pipeline ETL |
| Inspeção do Esquema | 14-16 | Análise do esquema bruto |
| Padronização | 17-19 | Normalização de nomes de colunas |
| Limpeza e Conversão | 20-22 | Conversão de tipos, trim, CNPJ, datas |
| Criação de Indicadores | 23-24 | Cálculo de cotas ativas, inadimplência |
| Qualidade de Dados | 25-35 | Testes de completude, unicidade, validade |
| Modelagem e Persistência | 36-44 | Tabelas Bronze, Silver e Gold |
| Análise de Dados | 45-84 | Consultas SQL, estatísticas e correlações |
| Segurança e Metadados | 85-86 | Ética, privacidade, metadados e linhagem |
| Autoavaliação | 87-91 | Objetivos, dificuldades, limitações, trabalhos futuros, conclusão |

> **Script de referência:** O notebook completo está disponível no GitHub: https://github.com/prchiquito83/mvp-consorcios-imobiliarios

###Inspeção do Esquema (em Python)

In [0]:
df_grupos_bruto.printSchema()

root
 |-- #Nome_da_Administradora: string (nullable = true)
 |-- CNPJ_da_Administradora: string (nullable = true)
 |-- Data_base: string (nullable = true)
 |-- Código_do_grupo: string (nullable = true)
 |-- Código_do_segmento: string (nullable = true)
 |-- Número_da_assembléia_geral_ordinária: string (nullable = true)
 |-- Valor_médio_do_bem: string (nullable = true)
 |-- Índice_de_correção: string (nullable = true)
 |-- Taxa_de_administração: string (nullable = true)
 |-- Prazo_do_grupo_em_meses: string (nullable = true)
 |-- Quantidade_de_cotas_ativas_em_dia: string (nullable = true)
 |-- Quantidade_de_cotas_ativas_contempladas_inadimplentes: string (nullable = true)
 |-- Quantidade_de_cotas_ativas_não_contempladas_inadimplentes: string (nullable = true)
 |-- Quantidade_de_cotas_ativas_contempladas_no_mês: string (nullable = true)
 |-- Quantidade_de_cotas_excluídas: string (nullable = true)
 |-- Quantidade_de_cotas_ativas_quitadas: string (nullable = true)
 |-- Quantidade_de_cotas_at

### Resultado da inspeção inicial
A camada bruta foi lida integralmente como texto para preservar os dados originais. Essa decisão como comentado anteriormente evita a perda de zeros à esquerda no CNPJ e permite controlar explicitamente a conversão dos valores monetários e percentuais que utilizam vírgula como separador decimal.

A próxima etapa aplicará padronização dos nomes das colunas, remoção de espaços, conversão de tipos e criação dos indicadores analíticos.

###Padronização dos Nomes das Colunas (em Python)

In [0]:
import re                       #Importa o módulo de expressões regulares (regex)
import unicodedata              # Importa funções para tratamento de caracteres Unicode (acentos, símbolos etc.)
def normalizar_nome_coluna(nome):
    """
    Converte um nome de coluna para uma forma simples:
    - remove acentos;
    - troca caracteres especiais por underscore;
    - converte para letras minúsculas.
    """
    nome = nome.lstrip("#")                                 # Remove o caractere "#" do início da string, se existir
    nome = unicodedata.normalize("NFKD", nome)              # Separa os caracteres dos seus acentos
    nome = nome.encode("ascii", "ignore").decode("ascii")   # Remove os caracteres não ASCII (acentos)
    nome = re.sub(r"[^a-zA-Z0-9]+", "_", nome)              # Substitui qualquer sequência de caracteres que não sejam letras ou números por "_"
    nome = nome.strip("_").lower()                          # Remove "_" do início e fim, e converte para minúsculas
    return nome                                             # Retorna o nome padronizado

# Para cada coluna do DataFrame original mantém os dados, mas altera o nome da coluna
colunas_normalizadas = [F.col(coluna).alias(normalizar_nome_coluna(coluna))
for coluna in df_grupos_bruto.columns
]

df_grupos_padrao = df_grupos_bruto.select(*colunas_normalizadas)

print(df_grupos_padrao.columns)
display(df_grupos_padrao.limit(5))

['nome_da_administradora', 'cnpj_da_administradora', 'data_base', 'codigo_do_grupo', 'codigo_do_segmento', 'numero_da_assembleia_geral_ordinaria', 'valor_medio_do_bem', 'indice_de_correcao', 'taxa_de_administracao', 'prazo_do_grupo_em_meses', 'quantidade_de_cotas_ativas_em_dia', 'quantidade_de_cotas_ativas_contempladas_inadimplentes', 'quantidade_de_cotas_ativas_nao_contempladas_inadimplentes', 'quantidade_de_cotas_ativas_contempladas_no_mes', 'quantidade_de_cotas_excluidas', 'quantidade_de_cotas_ativas_quitadas', 'quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao']


nome_da_administradora,cnpj_da_administradora,data_base,codigo_do_grupo,codigo_do_segmento,numero_da_assembleia_geral_ordinaria,valor_medio_do_bem,indice_de_correcao,taxa_de_administracao,prazo_do_grupo_em_meses,quantidade_de_cotas_ativas_em_dia,quantidade_de_cotas_ativas_contempladas_inadimplentes,quantidade_de_cotas_ativas_nao_contempladas_inadimplentes,quantidade_de_cotas_ativas_contempladas_no_mes,quantidade_de_cotas_excluidas,quantidade_de_cotas_ativas_quitadas,quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00057,1,194,"248144,39",3,"20,62128",200,699,48,0,0,2975,459,52
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00058,1,191,"566907,83",3,"18,09440",200,658,39,0,0,2130,346,83
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00061,1,190,"212069,65",3,"21,28078",200,749,51,0,0,2621,460,50
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00062,1,189,"297828,75",3,"18,02103",200,763,30,1,8,2268,371,149
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202607,00063,1,189,"252281,48",3,"10,91300",200,583,17,0,0,655,471,6


Alguns nomes técnicos das colunas sofreram boa alteração, mas nada que impeça o seu reconhecimento. A remoção de acentos não significa alterar o conteúdo do dado. Estamos apenas padronizando os nomes técnicos das colunas para facilitar o uso de SQL e Python

### Limpeza e Conversão dos Tipos

In [0]:
# Lista das colunas que devem ser convertidas para números inteiros (tipo long).
colunas_inteiras = [
    "codigo_do_segmento",                                           # Código do segmento do consórcio
    "numero_da_assembleia_geral_ordinaria",                         # Número da AGO (Assembleia Geral Ordinária)
    "prazo_do_grupo_em_meses",                                      # Prazo total do grupo em meses
    "quantidade_de_cotas_ativas_em_dia",                            # Total de cotas ativas no dia de referência
    "quantidade_de_cotas_ativas_contempladas_inadimplentes",        # Cotas contempladas com inadimplência
    "quantidade_de_cotas_ativas_nao_contempladas_inadimplentes",    # Cotas não contempladas inadimplentes
    "quantidade_de_cotas_ativas_contempladas_no_mes",               # Cotas contempladas durante o mês
    "quantidade_de_cotas_excluidas",                                # Quantidade de cotas excluídas
    "quantidade_de_cotas_ativas_quitadas",                          # Cotas já quitadas
    "quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao" # Cotas com crédito ainda não utilizado
]

# Lista das colunas que contêm valores numéricos decimais.
colunas_decimais = [
    "valor_medio_do_bem",                                           # Valor médio do bem referenciado pelo grupo
    "taxa_de_administracao"                                         # Taxa de administração cobrada pela administradora
]

# Utiliza o DataFrame previamente padronizado como base para o tratamento
df_grupos = df_grupos_padrao

# Limpeza e transformação das colunas textuais
df_grupos = (
    df_grupos
    .withColumn(                            # Remove espaços em branco no início e no fim do nome da administradora
        "nome_da_administradora",
        F.trim(F.col("nome_da_administradora"))
    )


    .withColumn(                            # Remove espaços do CNPJ e completa com zeros à esquerda até atingir 8 caracteres
        "cnpj_da_administradora",
    F.lpad(F.trim(F.col("cnpj_da_administradora")), 8, "0")
    )


    .withColumn(                                # Remove espaços extras do código do grupo
        "codigo_do_grupo",
        F.trim(F.col("codigo_do_grupo"))
    )


    .withColumn(                                    # Cria uma coluna temporária contendo o texto da data já limpo.
        "data_base_texto",
        F.trim(F.col("data_base"))
    )

# Converte uma data no formato AAAAMM para uma data Spark AAAA-MM-01
    .withColumn(
        "data_base",
        F.to_date(
            F.concat(F.substring("data_base_texto", 1, 4),   # Extrai o ano
                F.lit("-"),
                F.substring("data_base_texto", 5, 2),   # Extrai o mês
                F.lit("-01")                            # Define o primeiro dia do mês
            )
        )
    )
)

# Converte todas as colunas definidas na lista para tipo inteiro (long).
for coluna in colunas_inteiras:
    df_grupos = df_grupos.withColumn(
        coluna,F.trim(F.col(coluna)).cast("long"))# Remove espaços e converte para long

# Converte todas as colunas decimais.
for coluna in colunas_decimais:
    df_grupos = df_grupos.withColumn(
        coluna,F.regexp_replace(
        F.trim(F.col(coluna)),          # Remove espaços em branco.
        ",",                            # Localiza vírgulas decimais.
        "."                             # Substitui por ponto decimal.
        ).cast("double")                # Converte para número decimal (double).
    )

# Mantém o índice de correção como texto categórico.
df_grupos = df_grupos.withColumn(
    "indice_de_correcao",
    F.trim(F.col("indice_de_correcao")) # Apenas remove espaços extras.
)

# Exibe as 10 primeiras linhas do DataFrame tratado.
display(df_grupos.limit(10))

# Mostra a estrutura final do DataFrame (nomes das colunas e tipos de dados).
df_grupos.printSchema()

nome_da_administradora,cnpj_da_administradora,data_base,codigo_do_grupo,codigo_do_segmento,numero_da_assembleia_geral_ordinaria,valor_medio_do_bem,indice_de_correcao,taxa_de_administracao,prazo_do_grupo_em_meses,quantidade_de_cotas_ativas_em_dia,quantidade_de_cotas_ativas_contempladas_inadimplentes,quantidade_de_cotas_ativas_nao_contempladas_inadimplentes,quantidade_de_cotas_ativas_contempladas_no_mes,quantidade_de_cotas_excluidas,quantidade_de_cotas_ativas_quitadas,quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao,data_base_texto
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00057,1,194,248144.39,3,20.62128,200,699,48,0,0,2975,459,52,202607
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00058,1,191,566907.83,3,18.0944,200,658,39,0,0,2130,346,83,202607
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00061,1,190,212069.65,3,21.28078,200,749,51,0,0,2621,460,50,202607
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00062,1,189,297828.75,3,18.02103,200,763,30,1,8,2268,371,149,202607
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00063,1,189,252281.48,3,10.913,200,583,17,0,0,655,471,6,202607
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00064,1,189,219680.78,3,21.02703,200,769,30,0,0,2891,456,36,202607
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00065,1,188,252232.75,3,10.80283,200,570,30,0,0,769,435,15,202607
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00066,1,187,228547.51,3,20.12428,200,752,40,3,0,2853,403,76,202607
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00067,1,184,297777.39,3,19.34876,200,647,35,1,4,2205,330,71,202607
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00068,1,186,203130.41,3,21.39913,200,755,44,1,0,3362,448,67,202607


root
 |-- nome_da_administradora: string (nullable = true)
 |-- cnpj_da_administradora: string (nullable = true)
 |-- data_base: date (nullable = true)
 |-- codigo_do_grupo: string (nullable = true)
 |-- codigo_do_segmento: long (nullable = true)
 |-- numero_da_assembleia_geral_ordinaria: long (nullable = true)
 |-- valor_medio_do_bem: double (nullable = true)
 |-- indice_de_correcao: string (nullable = true)
 |-- taxa_de_administracao: double (nullable = true)
 |-- prazo_do_grupo_em_meses: long (nullable = true)
 |-- quantidade_de_cotas_ativas_em_dia: long (nullable = true)
 |-- quantidade_de_cotas_ativas_contempladas_inadimplentes: long (nullable = true)
 |-- quantidade_de_cotas_ativas_nao_contempladas_inadimplentes: long (nullable = true)
 |-- quantidade_de_cotas_ativas_contempladas_no_mes: long (nullable = true)
 |-- quantidade_de_cotas_excluidas: long (nullable = true)
 |-- quantidade_de_cotas_ativas_quitadas: long (nullable = true)
 |-- quantidade_de_cotas_ativas_com_credito_pend

De maneira sumarizada, a limpeza foi realizada nos seguintes aspectos:
- 'trim' removeu espaços antes e depois do texto.
- 'lpad(...,8,"0")' garante oito posições para a razi do CNPJ (não fiz verficações extras para o CNPJ).
- regexp_replace troca vírgula por ponto.
- cast("double") transforma texto em número decimal.
- to_date transforma o período em uma data válida usando o primeiro dia do mês.
- Como o relacional relacional trabalha com domínios é importante definir de maneira correta o tipo de cada coluna. É ainda uma forma de impor integridade de domínio.

###Criação dos Indicadores

**Definições**
>**Cotas ativas consideradas no indicador:**
>
>Cotas em dia + cotas contempladas inadimplentes + cotas não contempladas inadimplentes

Essa composição está alinhada ao dicionário, que utiliza essas três quantidades para representar a base de cotas ativas e para ponderar a taxa de administração consolidada.

>**Cotas inadimplentes:**
>
>Cotas contempladas inadimplentes + cotas não contempladas inadimplentes

>**Taxa inadimplência:**
>
>cotas inadimplentes / cotas ativas


In [0]:
df_grupos = (
    df_grupos
        .withColumn(
            "quantidade_cotas_inadimplentes",
            F.col("quantidade_de_cotas_ativas_contempladas_inadimplentes")
            + F.col("quantidade_de_cotas_ativas_nao_contempladas_inadimplentes")
        )
        .withColumn(
            "quantidade_cotas_ativas_calculada",
            F.col("quantidade_de_cotas_ativas_em_dia")
            + F.col("quantidade_de_cotas_ativas_contempladas_inadimplentes")
            + F.col("quantidade_de_cotas_ativas_nao_contempladas_inadimplentes")
        )
        .withColumn(
            "taxa_inadimplencia",
            F.when(
                F.col("quantidade_cotas_ativas_calculada") > 0,
                F.col("quantidade_cotas_inadimplentes")
                / F.col("quantidade_cotas_ativas_calculada")
            )
        )
        .withColumn(
            "taxa_em_dia",
            F.when(
                F.col("quantidade_cotas_ativas_calculada") > 0,
                F.col("quantidade_de_cotas_ativas_em_dia")
                / F.col("quantidade_cotas_ativas_calculada")
            )
        )        
)
display(
    df_grupos.select(
        "nome_da_administradora",
        "codigo_do_grupo",
        "quantidade_cotas_ativas_calculada",
        "quantidade_cotas_inadimplentes",
        "taxa_inadimplencia"
    ).limit(20)
)

nome_da_administradora,codigo_do_grupo,quantidade_cotas_ativas_calculada,quantidade_cotas_inadimplentes,taxa_inadimplencia
ITAÚ ADM DE CONSÓRCIOS LTDA,00057,747,48,0.0642570281124498
ITAÚ ADM DE CONSÓRCIOS LTDA,00058,697,39,0.05595408895265423
ITAÚ ADM DE CONSÓRCIOS LTDA,00061,800,51,0.06375
ITAÚ ADM DE CONSÓRCIOS LTDA,00062,794,31,0.03904282115869018
ITAÚ ADM DE CONSÓRCIOS LTDA,00063,600,17,0.028333333333333332
ITAÚ ADM DE CONSÓRCIOS LTDA,00064,799,30,0.03754693366708386
ITAÚ ADM DE CONSÓRCIOS LTDA,00065,600,30,0.05
ITAÚ ADM DE CONSÓRCIOS LTDA,00066,795,43,0.05408805031446541
ITAÚ ADM DE CONSÓRCIOS LTDA,00067,683,36,0.0527086383601757
ITAÚ ADM DE CONSÓRCIOS LTDA,00068,800,45,0.05625


## Etapa 4.5 — Qualidade de Dados

A qualidade dos dados depende do que se conhece dos metadados, como a estrutura, os tipos, os domínios, as chaves, a granularidade, a temporalidade, a origem e as transformações realizadas.
Para fins de exercício do MVP iremos analisar as seguintes dimensões:

- completude;
- unicidade;
- validade;
- consistência;
- conformidade;
- plausibilidade.

In [0]:
from functools import reduce

# Conta o total de linhas do DataFrame para servir de base no cálculo de percentuais
total_linhas = df_grupos.count()

# Lista que armazenará, para cada coluna, o nome, a quantidade de nulos e o percentual de nulos
resultado_nulos = []

# Percorre todas as colunas do DataFrame
for coluna in df_grupos.columns:
    # Conta quantas linhas possuem valor nulo na coluna atual
    quantidade_nulos = df_grupos.filter(F.col(coluna).isNull()).count()
    # Adiciona à lista uma tupla com: nome da coluna, quantidade de nulos e percentual de nulos
    resultado_nulos.append(
        (coluna, quantidade_nulos, quantidade_nulos / total_linhas)
    )

# Cria um DataFrame Spark a partir da lista de tuplas contendo os resultados
df_nulos = spark.createDataFrame(
    resultado_nulos,
    ["coluna", "quantidade_nulos", "percentual_nulos"]
    )

# Exibe o DataFrame ordenado do maior para o menor percentual de nulos
display(df_nulos.orderBy(F.desc("percentual_nulos")))

coluna,quantidade_nulos,percentual_nulos
nome_da_administradora,0,0.0
cnpj_da_administradora,0,0.0
data_base,0,0.0
codigo_do_grupo,0,0.0
codigo_do_segmento,0,0.0
numero_da_assembleia_geral_ordinaria,0,0.0
valor_medio_do_bem,0,0.0
indice_de_correcao,0,0.0
taxa_de_administracao,0,0.0
prazo_do_grupo_em_meses,0,0.0


###Avaliação da Duplicidade da Chave de Negócio

Foi adotada uma chave composta por CNPJ_da_Administradora, Data_base, Código_do_grupo e Código_do_segmento, pois esses atributos representam a granularidade dos dados. Entretanto, essa chave foi tratada como uma hipótese inicial, podendo ser reduzida após testes de unicidade. Caso o Código_do_segmento não contribua para diferenciar registros, a chave mínima passaria a ser CNPJ_da_Administradora + Data_base + Código_do_grupo, atendendo ao princípio de [minimalidade](url) das chaves candidatas.
Logo a chave de negócio esperada inicialmente para fins do MVP é:
>CNPJ + data-base + código do grupo + código do segmento

In [0]:
# Teste de unicidade da chave de negócio
# O objetivo deste teste é verificar se existem registros duplicados, ou seja, mais de uma linha com a mesma combinação desses quatro campos.
# Caso existam duplicidades, a chave precisará ser revisada (ampliada ou substituída) para garantir a integridade da granularidade dos dados.

duplicidades = (
    df_grupos
    .groupBy(
        "cnpj_da_administradora",   
        "data_base",                 
        "codigo_do_grupo",           
        "codigo_do_segmento"        # No teste com apenas essa porção de dados, esse atributo não é necessário, pois não há valores repetidos, agora se usarem mais dados, acredito ser um atributo importante para composição da chave 
    )
    .count()                        # Conta quantas linhas existem em cada combinação
    .filter(F.col("count") > 1)     # Mantém apenas as combinações com mais de 1 ocorrência (duplicadas)
)

# Exibe a quantidade total de combinações duplicadas encontradas.
# Se o resultado for 0 (zero), a chave de negócio é única e válida.
print("Quantidade de combinações duplicadas:", duplicidades.count())

# Exibe as combinações duplicadas para inspeção visual.
display(duplicidades)

Quantidade de combinações duplicadas: 0


cnpj_da_administradora,data_base,codigo_do_grupo,codigo_do_segmento,count


###Avaliação de Valores Impossíveis


In [0]:
regras_qualidade = (
    df_grupos
    .select(
        F.sum(
            F.when(F.col("cnpj_da_administradora").isNull(), 1).otherwise(0)
        ).alias("cnpj_nulo"),

        F.sum(
            F.when(F.length("cnpj_da_administradora") != 8, 1).otherwise(0)
        ).alias("cnpj_tamanho_invalido"),

        F.sum(
            F.when(F.col("data_base").isNull(), 1).otherwise(0)
        ).alias("data_base_invalida"),

        F.sum(
            F.when(F.col("codigo_do_segmento") != 1, 1).otherwise(0)
        ).alias("segmento_diferente_de_imoveis"),

        F.sum(
            F.when(F.col("valor_medio_do_bem") < 0, 1).otherwise(0)
        ).alias("valor_medio_negativo"),

        F.sum(
            F.when(F.col("taxa_de_administracao") < 0, 1).otherwise(0)
        ).alias("taxa_administracao_negativa"),

        F.sum(
            F.when(F.col("prazo_do_grupo_em_meses") <= 0, 1).otherwise(0)
        ).alias("prazo_invalido"),

        F.sum(
            F.when(
                (F.col("taxa_inadimplencia") < 0)
                |(F.col("taxa_inadimplencia") > 1),
                1
            ).otherwise(0)
        ).alias("taxa_inadimplencia_invalida")
    )
)

display(regras_qualidade)

cnpj_nulo,cnpj_tamanho_invalido,data_base_invalida,segmento_diferente_de_imoveis,valor_medio_negativo,taxa_administracao_negativa,prazo_invalido,taxa_inadimplencia_invalida
0,0,0,0,0,0,0,0


###Avaliação dos Zeros x Nulos

In [0]:
# Conta a quantidade de valores iguais a zero em cada coluna inteira.
display(
    df_grupos.select(
        [
        F.sum(F.when(F.col(c) == 0, 1).otherwise(0)).alias(c)
            for c in colunas_inteiras
        ]
    )
)

codigo_do_segmento,numero_da_assembleia_geral_ordinaria,prazo_do_grupo_em_meses,quantidade_de_cotas_ativas_em_dia,quantidade_de_cotas_ativas_contempladas_inadimplentes,quantidade_de_cotas_ativas_nao_contempladas_inadimplentes,quantidade_de_cotas_ativas_contempladas_no_mes,quantidade_de_cotas_excluidas,quantidade_de_cotas_ativas_quitadas,quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao
0,0,0,0,367,170,129,18,639,22


In [0]:
# Conta a quantidade de valores iguais a Null em cada coluna inteira.
display(
    df_grupos.select(
        [
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in colunas_inteiras
        ]
    )
)

codigo_do_segmento,numero_da_assembleia_geral_ordinaria,prazo_do_grupo_em_meses,quantidade_de_cotas_ativas_em_dia,quantidade_de_cotas_ativas_contempladas_inadimplentes,quantidade_de_cotas_ativas_nao_contempladas_inadimplentes,quantidade_de_cotas_ativas_contempladas_no_mes,quantidade_de_cotas_excluidas,quantidade_de_cotas_ativas_quitadas,quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao
0,0,0,0,0,0,0,0,0,0


Foi realizada uma verificação da ocorrência de valores iguais a zero nas principais métricas quantitativas. Observou-se que diversas colunas apresentam registros com valor zero, especialmente relacionadas à inadimplência, contemplação, exclusão, quitação e crédito pendente. Esses valores foram mantidos, pois representam situações válidas do negócio, indicando a inexistência da ocorrência medida naquele grupo. Valores zero não foram tratados como dados ausentes, uma vez que, no modelo relacional, zero e NULL possuem significados distintos: zero representa uma quantidade conhecida igual a zero, enquanto NULL representa informação ausente, desconhecida ou inaplicável.

Não houve nenhuma ocorrência de valores nulos.

### Avaliação da Qualidade dos Dados

Foram avaliadas completude, unicidade, validade, consistência e conformidade.

A chave de negócio adotada foi composta pela raiz do CNPJ da administradora, data-base, código do grupo e código do segmento. Essa composição representa a granularidade esperada da fonte. (Não foram identificados duplicidade nas chaves de negócio).

Os testes verificaram:

- campos obrigatórios nulos;
- CNPJ com tamanho diferente de oito caracteres;
- datas que não puderam ser convertidas;
- presença de segmentos diferentes de bens imóveis;
- valores monetários negativos;
- taxas de administração negativas;
- prazos inválidos;
- taxas de inadimplência fora do intervalo entre zero e um;
- combinações duplicadas da chave de negócio.

Valores iguais a zero foram preservados, pois zero é um dado válido e não deve ser confundido com ausência de informação.

## Etapa 4.3 — Modelagem e Catálogo de Dados

A modelagem de dados seguiu a arquitetura de camadas **Bronze → Silver → Gold**, persistidas no Unity Catalog (`workspace.mvp_consorcios`) no formato Delta Lake.

**Catálogo de tabelas criadas:**

| Tabela | Camada | Descrição | Granularidade |
|---|---|---|---|
| `bronze_grupos_imoveis` | Bronze | Dados com colunas normalizadas, próximos ao formato original | Uma linha por grupo/segmento/data-base |
| `silver_grupos_imoveis` | Silver | Dados tratados: tipos convertidos, espaços removidos, datas e decimais padronizados | Uma linha por grupo/segmento/data-base |
| `gold_administradoras` | Gold | Indicadores agregados por administradora (cotas ativas, inadimplência, crédito pendente, taxa de administração ponderada) | Uma linha por administradora/data-base |
| `gold_grupos_risco` | Gold | Dados por grupo com indicadores calculados e classificações de risco e prazo | Uma linha por grupo/segmento/data-base |
| `vw_monitoramento_risco` | View | Visão de monitoramento filtrando apenas grupos com cotas ativas | Mesma granularidade da Gold de grupos |

As tabelas podem ser visualizadas no Catalog Explorer do Databricks em `workspace.mvp_consorcios`.

A captura de imagem abaixo foi feita para ilustrar as tabelas no Catalog Explorer do Databricks.

![image_1790458420444.png](./image_1790458420444.png "image_1790458420444.png")


### Armazenamento dos dados — Tabelas Bronze e Silver

In [0]:
# Persistência da camada bruta.
(
    df_grupos_padrao.write                                                   # DataFrame com colunas normalizadas (sem tratamento de tipos)
    .format("delta")                                                         # Formato Delta para a tabela
    .mode("overwrite")                                                       # Sobrescreve os dados existentes
    .option("overwriteSchema", "true")                                       # Permite alterar o schema ao sobrescrever
    .saveAsTable("workspace.mvp_consorcios.bronze_grupos_imoveis")           # Salva na camada Bronze do catálogo workspace
)
# Persistência da camada tratada.
(
    df_grupos.write                                                          # DataFrame já tratado com tipos convertidos e indicadores
    .format("delta")                                                         # Formato Delta para a tabela
    .mode("overwrite")                                                       # Sobrescreve os dados existentes
    .option("overwriteSchema", "true")                                       # Permite alterar o schema ao sobrescrever
    .saveAsTable("workspace.mvp_consorcios.silver_grupos_imoveis")           # Salva na camada Silver do catálogo workspace
)

print("Tabelas bronze e silver criadas.")                                   # Confirma a criação das tabelas

Tabelas bronze e silver criadas.


Os dados foram armazenados em duas camadas que podem ser visualizadas dentro do "Catalog". A camada Bronze preserva os dados próximos ao formato original recebido, permitindo rastreabilidade e reconstrução das transformações. A camada Silver contém os dados tratados, padronizados e convertidos para os tipos adequados, servindo como base para a criação dos indicadores analíticos e das consultas do projeto. Essa separação facilita a qualidade dos dados, a linhagem das transformações e a manutenção do processo ETL.



###Construção da Tabela Gold por Administradora

In [0]:
df_administradoras = (
    df_grupos
    .groupBy(
        "cnpj_da_administradora",      # Agrupa por CNPJ da administradora
        "nome_da_administradora",      # e pelo nome da administradora
        "data_base"                   # e pela data-base de referência
    )
    .agg(
        F.countDistinct("codigo_do_grupo").alias("quantidade_grupos"),                          # Conta grupos distintos
        F.sum("quantidade_cotas_ativas_calculada").alias("quantidade_cotas_ativas"),              # Soma total de cotas ativas
        F.sum("quantidade_cotas_inadimplentes").alias("quantidade_cotas_inadimplentes"),          # Soma cotas inadimplentes
        F.sum("quantidade_de_cotas_excluidas").alias("quantidade_cotas_excluidas"),                # Soma cotas excluídas
        F.sum("quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao").alias("quantidade_credito_pendente"),  # Soma cotas com crédito pendente
        F.sum("quantidade_de_cotas_ativas_contempladas_no_mes").alias("quantidade_contempladas_mes"),  # Soma cotas contempladas no mês
        F.avg("valor_medio_do_bem").alias("valor_medio_bem_media_simples"),                        # Média simples do valor médio do bem
        (
            F.sum(
                F.col("taxa_de_administracao")
                * F.col("quantidade_cotas_ativas_calculada")
            )
            / F.sum("quantidade_cotas_ativas_calculada")
        ).alias("taxa_administracao_ponderada")                                                    # Média ponderada da taxa de administração
    )
    .withColumn(
        "taxa_inadimplencia",
        F.when(
            F.col("quantidade_cotas_ativas") > 0,                    # Se há cotas ativas
            F.col("quantidade_cotas_inadimplentes")
            / F.col("quantidade_cotas_ativas")                       # Calcula a taxa de inadimplência
        )
    )
    .withColumn(
        "taxa_credito_pendente",
        F.when(
            F.col("quantidade_cotas_ativas") > 0,                    # Se há cotas ativas
            F.col("quantidade_credito_pendente")
            / F.col("quantidade_cotas_ativas")                       # Calcula a taxa de crédito pendente
        )
    )
)
(
    df_administradoras.write
    .format("delta")                                               # Formato Delta
    .mode("overwrite")                                             # Sobrescreve os dados
    .option("overwriteSchema", "true")                             # Permite alterar o schema
    .saveAsTable("workspace.mvp_consorcios.gold_administradoras")  # Salva na camada Gold
)
display(df_administradoras.limit(20))                              # Exibe as 20 primeiras linhas

cnpj_da_administradora,nome_da_administradora,data_base,quantidade_grupos,quantidade_cotas_ativas,quantidade_cotas_inadimplentes,quantidade_cotas_excluidas,quantidade_credito_pendente,quantidade_contempladas_mes,valor_medio_bem_media_simples,taxa_administracao_ponderada,taxa_inadimplencia,taxa_credito_pendente
73516106,HS ADM CONS S.A.,2026-07-01,152,235649,16794,211736,12949,835,257379.45243421054,21.923970713476393,0.07126701152986009,0.05495037110278422
23767155,RECON ADM. DE CONSORCIOS LTDA,2026-07-01,5,7606,1292,30767,697,50,186077.318,22.61686778595845,0.16986589534577964,0.09163818038390745
06181431,BP,2026-07-01,5,2550,799,6846,94,8,193133.2,24.644216588235295,0.31333333333333335,0.03686274509803922
62534158,FF ADM CONS LTDA.,2026-07-01,1,273,22,202,5,1,223168.0,21.2051,0.08058608058608059,0.018315018315018316
28904092,CONSÓRCIO RESERVA,2026-07-01,3,1639,402,14042,59,6,268684.8333333333,24.349907907260526,0.24527150701647346,0.03599755948749237
84911098,ADEMICON ADM CONS S.A.,2026-07-01,145,572105,82413,296900,9402,1076,214485.0550344828,24.040866013738743,0.14405222817489796,0.01643404619781334
59956185,TRADIÇÃO ADM CONS. LTDA.,2026-07-01,4,4787,877,11894,172,14,226426.335,20.855587808648426,0.18320451222059744,0.035930645498224355
13170428,CIMAVEL ADM CONS,2026-07-01,1,156,19,1176,32,1,130534.14,19.9051,0.12179487179487179,0.20512820512820512
42735881,EVOY ADM CONS LTDA.,2026-07-01,1,2551,471,7591,54,2,281143.53,25.08163,0.18463347706781655,0.021168169345354763
06044551,CONSORCIO GAZIN,2026-07-01,10,11999,2257,14621,549,38,169472.883,22.41326134344529,0.18809900825068757,0.04575381281773481


A ponderação da taxa de administração pela quantidade de cotas ativas me pareceu compatível com a regra descrita no dicionário de métricas.

###Inclusão de Sistemas de Classificação - Risco e Prazo

Se é proposto a seguinte regra para avaliar os ricos causados pela inadimplência dos consórcios:
- Baixo: Inadimplência menor que 5%
- Moderado: de 5% até menos de 10%
- Alto: de 10% até menos de 20%
- Muito alto: 20% ou mais

Esses limites não vieram da fonte de dados nem de norma apresentada pelo Banco Central. Essa classificação dos parâmetros foi criada para fins de estudo do MVP e responder as perguntas de negócio.

Além disso iremos classificar os grupos de acordo com os prazos em meses:
- 120 | 180 

In [0]:
df_grupos_risco = (
    df_grupos
    .withColumn(
        "faixa_inadimplencia",
        F.when(F.col("taxa_inadimplencia").isNull(), "Sem base de cálculo")
        .when(F.col("taxa_inadimplencia") < 0.05, "Baixo")
        .when(F.col("taxa_inadimplencia") < 0.10, "Moderado")
        .when(F.col("taxa_inadimplencia") < 0.20, "Alto")
        .otherwise("Muito alto")
    )
    .withColumn(
        "faixa_prazo",
        F.when(F.col("prazo_do_grupo_em_meses") <= 120, "Até 120 meses")
        .when(F.col("prazo_do_grupo_em_meses") <= 180, "121 a 180 meses")
        .when(F.col("prazo_do_grupo_em_meses") <= 240, "181 a 240 meses")
        .otherwise("Acima de 240 meses")
    )
)
(
df_grupos_risco.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.mvp_consorcios.gold_grupos_risco")
)
display(df_grupos_risco.limit(20))

nome_da_administradora,cnpj_da_administradora,data_base,codigo_do_grupo,codigo_do_segmento,numero_da_assembleia_geral_ordinaria,valor_medio_do_bem,indice_de_correcao,taxa_de_administracao,prazo_do_grupo_em_meses,quantidade_de_cotas_ativas_em_dia,quantidade_de_cotas_ativas_contempladas_inadimplentes,quantidade_de_cotas_ativas_nao_contempladas_inadimplentes,quantidade_de_cotas_ativas_contempladas_no_mes,quantidade_de_cotas_excluidas,quantidade_de_cotas_ativas_quitadas,quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao,data_base_texto,quantidade_cotas_inadimplentes,quantidade_cotas_ativas_calculada,taxa_inadimplencia,taxa_em_dia,faixa_inadimplencia,faixa_prazo
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00057,1,194,248144.39,3,20.62128,200,699,48,0,0,2975,459,52,202607,48,747,0.0642570281124498,0.9357429718875502,Moderado,181 a 240 meses
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00058,1,191,566907.83,3,18.0944,200,658,39,0,0,2130,346,83,202607,39,697,0.05595408895265423,0.9440459110473458,Moderado,181 a 240 meses
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00061,1,190,212069.65,3,21.28078,200,749,51,0,0,2621,460,50,202607,51,800,0.06375,0.93625,Moderado,181 a 240 meses
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00062,1,189,297828.75,3,18.02103,200,763,30,1,8,2268,371,149,202607,31,794,0.03904282115869018,0.9609571788413098,Baixo,181 a 240 meses
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00063,1,189,252281.48,3,10.913,200,583,17,0,0,655,471,6,202607,17,600,0.028333333333333332,0.9716666666666667,Baixo,181 a 240 meses
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00064,1,189,219680.78,3,21.02703,200,769,30,0,0,2891,456,36,202607,30,799,0.03754693366708386,0.9624530663329162,Baixo,181 a 240 meses
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00065,1,188,252232.75,3,10.80283,200,570,30,0,0,769,435,15,202607,30,600,0.05,0.95,Moderado,181 a 240 meses
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00066,1,187,228547.51,3,20.12428,200,752,40,3,0,2853,403,76,202607,43,795,0.05408805031446541,0.9459119496855346,Moderado,181 a 240 meses
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00067,1,184,297777.39,3,19.34876,200,647,35,1,4,2205,330,71,202607,36,683,0.0527086383601757,0.9472913616398243,Moderado,181 a 240 meses
ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,2026-07-01,00068,1,186,203130.41,3,21.39913,200,755,44,1,0,3362,448,67,202607,45,800,0.05625,0.94375,Moderado,181 a 240 meses


## Etapa 4.5 — Análise de Dados

Nesta seção, as perguntas de negócio formuladas na Etapa 4.1 são respondidas por meio de consultas SQL sobre as tabelas Gold. As análises incluem ranking de administradoras, identificação de grupos críticos, distribuição por faixa de risco, correlações exploratórias e estatísticas descritivas.

### Consultas SQL

**Consulta 1 - Visão Geral da Camada Gold**

Verificação
> A estrutura da camada analítica foi criada corretamente e os indicadores estão disponíveis para análise?

Objetivo de visualizar os primeiros registros da tabela Gold para validar o carregamento dos dados.


In [0]:
%sql
SELECT *
FROM workspace.mvp_consorcios.gold_administradoras
LIMIT 20;

cnpj_da_administradora,nome_da_administradora,data_base,quantidade_grupos,quantidade_cotas_ativas,quantidade_cotas_inadimplentes,quantidade_cotas_excluidas,quantidade_credito_pendente,quantidade_contempladas_mes,valor_medio_bem_media_simples,taxa_administracao_ponderada,taxa_inadimplencia,taxa_credito_pendente
73516106,HS ADM CONS S.A.,2026-07-01,152,235649,16794,211736,12949,835,257379.45243421054,21.923970713476393,0.07126701152986009,0.05495037110278422
23767155,RECON ADM. DE CONSORCIOS LTDA,2026-07-01,5,7606,1292,30767,697,50,186077.318,22.61686778595845,0.16986589534577964,0.09163818038390745
06181431,BP,2026-07-01,5,2550,799,6846,94,8,193133.2,24.644216588235295,0.31333333333333335,0.03686274509803922
62534158,FF ADM CONS LTDA.,2026-07-01,1,273,22,202,5,1,223168.0,21.2051,0.08058608058608059,0.018315018315018316
28904092,CONSÓRCIO RESERVA,2026-07-01,3,1639,402,14042,59,6,268684.8333333333,24.349907907260526,0.24527150701647346,0.03599755948749237
84911098,ADEMICON ADM CONS S.A.,2026-07-01,145,572105,82413,296900,9402,1076,214485.0550344828,24.040866013738743,0.14405222817489796,0.01643404619781334
59956185,TRADIÇÃO ADM CONS. LTDA.,2026-07-01,4,4787,877,11894,172,14,226426.335,20.855587808648426,0.18320451222059744,0.035930645498224355
13170428,CIMAVEL ADM CONS,2026-07-01,1,156,19,1176,32,1,130534.14,19.9051,0.12179487179487179,0.20512820512820512
42735881,EVOY ADM CONS LTDA.,2026-07-01,1,2551,471,7591,54,2,281143.53,25.08163,0.18463347706781655,0.021168169345354763
06044551,CONSORCIO GAZIN,2026-07-01,10,11999,2257,14621,549,38,169472.883,22.41326134344529,0.18809900825068757,0.04575381281773481


Foi possível identificar uma linha por administradora e também visualizar os principais indicadores consolidados.

**Consulta 2 - Administradoras com Maior Quantidade de Grupos**

Pergunta de Negócio
> Quais administradoras possuem mais grupos imobiliários ativos?

Objetivo de identificar a concentração operacional entre as administradoras.

In [0]:
%sql
SELECT
nome_da_administradora,
quantidade_grupos
FROM workspace.mvp_consorcios.gold_administradoras
ORDER BY quantidade_grupos DESC
LIMIT 10;

nome_da_administradora,quantidade_grupos
BRADESCO CONS. LTDA.,438
PORTO SEGURO ADM. CONS. LTDA,422
ITAÚ ADM DE CONSÓRCIOS LTDA,263
HS ADM CONS S.A.,152
SANTANDER BRASIL ADM CONS LTDA,147
ADEMICON ADM CONS S.A.,145
RODOBENS ADM CONSORCIOS LTDA,137
SICOOB ADM CONS LTDA.,121
ADM CONS SICREDI LTDA,119
EMBRACON ADM CONS S.A.,115


Obtivemos uma resposta com as administradoras com maior quantidade de grupos em ordem descrescente. As administradoras com maior quantidade de grupos tendem a possuir maior escala operacional.

**Consulta 3 - Grupos com Maior Inadimplência**

Pergunta de Negócio
> Quais grupos imobiliários apresentam os maiores índices de inadimplência?

Objetivo de identificar grupos de maior atenção operacional.

In [0]:
%sql
SELECT
nome_da_administradora,
codigo_do_grupo,
quantidade_cotas_ativas_calculada,
quantidade_cotas_inadimplentes,
ROUND(taxa_inadimplencia * 100, 2) AS inadimplencia_percentual,
quantidade_de_cotas_excluidas
FROM workspace.mvp_consorcios.gold_grupos_risco
WHERE quantidade_cotas_ativas_calculada >= 100
ORDER BY taxa_inadimplencia DESC
LIMIT 20;

nome_da_administradora,codigo_do_grupo,quantidade_cotas_ativas_calculada,quantidade_cotas_inadimplentes,inadimplencia_percentual,quantidade_de_cotas_excluidas
KSK ADM CONS LTDA.,00707,313,275,87.86,2
KSK ADM CONS LTDA.,00706,4308,3754,87.14,1110
KSK ADM CONS LTDA.,00703,972,721,74.18,3575
KSK ADM CONS LTDA.,00702,841,585,69.56,4553
KSK ADM CONS LTDA.,00704,1935,1310,67.7,5606
ALPHA ADM CONS LTDA,1004,1178,704,59.76,4448
PORTO SEGURO ADM. CONS. LTDA,I0387,900,509,56.56,3
ALPHA ADM CONS LTDA,S1005,4235,2207,52.11,3902
KSK ADM CONS LTDA.,00701,930,473,50.86,5766
ALPHA ADM CONS LTDA,S1007,2698,1278,47.37,2049


Os grupos apresentados no topo do ranking apresentam maior proporção de cotas inadimplentes e podem merecer investigação posterior.

**Consulta 4 - Taxa Consolidada por Administradora**

Pergunta de Negócio

>Quais administradoras apresentam maior inadimplência agregada?

Objetivo de comparar administradora ao invés de grupos individuais.

In [0]:
%sql
SELECT
nome_da_administradora,
quantidade_grupos,
quantidade_cotas_ativas,
quantidade_cotas_inadimplentes,
ROUND(taxa_inadimplencia * 100, 2) AS inadimplencia_percentual
FROM workspace.mvp_consorcios.gold_administradoras
WHERE quantidade_cotas_ativas >= 1000
ORDER BY taxa_inadimplencia DESC
LIMIT 20;

nome_da_administradora,quantidade_grupos,quantidade_cotas_ativas,quantidade_cotas_inadimplentes,inadimplencia_percentual
KSK ADM CONS LTDA.,7,10275,7492,72.91
ALPHA ADM CONS LTDA,5,13621,6338,46.53
BP,5,2550,799,31.33
CONSÓRCIO RESERVA,3,1639,402,24.53
COOPERATIVA ADM CONS ROMA (EX JOCKEY,8,4279,1015,23.72
ZEMA ADM. CONSÓRCIO LTDA,14,7879,1841,23.37
SPONCHIADO ADM CONSÓRCIOS LTDA.,4,1405,326,23.2
TRIÂNGULO ADM CONS S/A,4,2619,607,23.18
FARROUPILHA ADM CONS LTDA,3,2684,620,23.1
GLOBO ADM. CONS. LTDA.,5,1747,387,22.15


Databricks visualization. Run in Databricks to view.

O gráfico evidencia as administradoras com os maiores índices de inadimplência observados na base analisada. A visualização facilita a identificação de empresas que merecem investigação adicional quanto ao perfil das carteiras administradas, características dos grupos ou estratégias de cobrança adotadas.

Podemos verificar que justamente as administradoras com maiores taxas de inadimplência indicam a maior concetração de grupos problemáticos, conforme a consulta anterior.

**Consulta 5 - Inadimplência por Faixa de Prazo**

Pergunta de Negócio

>Existe diferença de inadimplência entre grupos de curto, médio e longo prazo?

Objetivo de investigar associação entre prazo do grupo e inadimplência.


In [0]:
%sql
SELECT
faixa_prazo,
COUNT(*) AS quantidade_grupos,
SUM(quantidade_cotas_ativas_calculada) AS cotas_ativas,
SUM(quantidade_cotas_inadimplentes) AS cotas_inadimplentes,
ROUND(100.0 * SUM(quantidade_cotas_inadimplentes)/NULLIF(SUM(quantidade_cotas_ativas_calculada), 0),2) AS inadimplencia_percentual
FROM workspace.mvp_consorcios.gold_grupos_risco
GROUP BY faixa_prazo
ORDER BY inadimplencia_percentual DESC;

faixa_prazo,quantidade_grupos,cotas_ativas,cotas_inadimplentes,inadimplencia_percentual
Acima de 240 meses,99,176445,23778,13.48
181 a 240 meses,1929,2398433,240724,10.04
121 a 180 meses,766,679050,47232,6.96
Até 120 meses,41,19044,1276,6.70


Databricks visualization. Run in Databricks to view.

Os resultados indicam uma correlação entre o prazo do grupo e a inadimplência: grupos com prazos mais curtos (até 180 meses) mantêm taxas de inadimplência semelhantes e mais controladas (~6.8%), enquanto grupos de prazo mais longo (acima de 180 meses) apresentam um crescimento significativo na inadimplência. Esse padrão sugere que grupos com prazos mais longos estão mais sujeitos a inadimplência, merecendo maior atenção e monitoramento por parte das administradoras. A faixa de 181 a 240 meses apresentou um crescimento próximo de 3% e a faixa acima de 240 meses praticamente dobrou a taxa de inadimplência em comparação aos grupos com faixa de prazo inferior a 180 meses.

Esse comportamento pode indicar que compromissos financeiros de longa duração estão associados a uma maior probabilidade de atraso ou não pagamento das parcelas. Quanto maior o horizonte temporal do consórcio, maior tende a ser a exposição dos participantes a mudanças em sua situação financeira, econômica ou profissional ao longo do tempo.

Além disso, embora a faixa "Acima de 240 meses" apresente a maior inadimplência, ela representa apenas 99 grupos, enquanto a faixa "181 a 240 meses" concentra a maior parte da população analisada, com 1.929 grupos e mais de 2,3 milhões de cotas ativas. Portanto, os resultados da faixa mais longa devem ser interpretados com cautela devido ao menor volume de observações.

**Consulta 6 - Inadimplência em comparação a Volume Operacional**

Pergunta de Negócio

>Entre as administradoras com grande volume operacional, quais possuem maior inadimplência?

Objetivo de filtrar apenas administradoras relevantes em termos de volume.

(Considerado que alto volume operacional =  uma administradora que opera uma quantidade de cotas ativas igual ou superior a 5000).

In [0]:
%sql
SELECT
nome_da_administradora,
COUNT(DISTINCT codigo_do_grupo) AS quantidade_grupos,
SUM(quantidade_cotas_ativas_calculada) AS cotas_ativas,
ROUND(
100.0 * SUM(quantidade_cotas_inadimplentes)
/ NULLIF(SUM(quantidade_cotas_ativas_calculada), 0),
2
) AS inadimplencia_percentual
FROM workspace.mvp_consorcios.gold_grupos_risco
GROUP BY nome_da_administradora
HAVING SUM(quantidade_cotas_ativas_calculada) >= 5000
ORDER BY inadimplencia_percentual DESC;


nome_da_administradora,quantidade_grupos,cotas_ativas,inadimplencia_percentual
KSK ADM CONS LTDA.,7,10275,72.91
ALPHA ADM CONS LTDA,5,13621,46.53
ZEMA ADM. CONSÓRCIO LTDA,14,7879,23.37
CONSORCIO GAZIN,10,11999,18.81
CANOPUS ADM CONSÓRCIOS,9,16176,17.77
MULTIMARCAS ADM.CONS.LTDA.,31,9745,17.32
RECON ADM. DE CONSORCIOS LTDA,5,7606,16.99
BAMAQ ADM CONS LTDA,3,6172,15.31
ADEMICON ADM CONS S.A.,145,572105,14.41
LUIZA ADM. CONS. LTDA,26,43230,13.51


Esta consulta restringe a análise às administradoras com alto volume operacional (≥ 5.000 cotas ativas), garantindo que a taxa de inadimplência seja calculada sobre uma base estatisticamente relevante. Ao filtrar por massa crítica, evitamos distorções provocadas por administradoras menores, onde poucas cotas inadimplentes podem inflar artificialmente o percentual.

Observa-se que as administradoras que já lideravam o ranking de inadimplência agregada (Consulta 4) permanecem no topo mesmo após a aplicação do filtro de volume. Isso indica que as altas taxas de inadimplência observadas **não são uma consequência do porte operacional** — ou seja, o problema não está associado ao tamanho da operação, mas possivelmente a fatores como perfil dos consorciados, qualidade da análise de crédito, gestão de cobrança ou características dos grupos administrados.

Essa constatação é relevante para a estratégia de mitigação de risco: reduzir o volume operacional por si só não resolve a inadimplência. As administradoras com maiores taxas precisam de ações direcionadas à gestão de risco e cobrança, independentemente do seu porte.

**Consulta 7 - Grupos com Maior Crédito Pendente de Utilização**

Pergunta de Negócio

>Quais grupos possuem maior quantidade de créditos contemplados ainda não utilizados?

Objetivo de identificar grupos que possuem contemplações concedidas mas cujo crédito permanece pendente.

In [0]:
%sql
SELECT
nome_da_administradora,
codigo_do_grupo,
quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao AS credito_pendente,
quantidade_cotas_ativas_calculada,
ROUND(100.0* quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao/ NULLIF(quantidade_cotas_ativas_calculada, 0),2) AS credito_pendente_percentual
FROM workspace.mvp_consorcios.gold_grupos_risco
WHERE quantidade_cotas_ativas_calculada >= 100
ORDER BY credito_pendente_percentual DESC
LIMIT 20;

nome_da_administradora,codigo_do_grupo,credito_pendente,quantidade_cotas_ativas_calculada,credito_pendente_percentual
BR CONSÓRCIOS ADM. CONS. LTDA.,MCA92,82,205,40.00
MULTIMARCAS ADM.CONS.LTDA.,00993,66,174,37.93
MULTIMARCAS ADM.CONS.LTDA.,02036,97,265,36.60
RODOBENS ADM CONSORCIOS LTDA,01910,44,133,33.08
ADM CONS SICREDI LTDA,70052,288,900,32.00
ADM CONS SICREDI LTDA,70053,190,596,31.88
ADM CONS SICREDI LTDA,70051,286,900,31.78
ADM CONS SICREDI LTDA,70061,283,895,31.62
ADM CONS SICREDI LTDA,70057,284,899,31.59
ADM CONS SICREDI LTDA,70056,280,900,31.11


Os resultados estão ordenados em ordem decrescente pelo percentual de crédito pendente em relação ao total de cotas ativas de cada grupo. Esse percentual (`credito_pendente_percentual`) permite comparar grupos de diferentes tamanhos de forma justa, evidenciando não apenas o volume absoluto de cotas com crédito pendente, mas a **proporção** dessas cotas dentro do grupo.

Grupos com percentuais elevados de crédito pendente merecem atenção, pois indicam que uma parcela significativa dos consorciados contemplados ainda não utilizou o crédito disponibilizado. Isso pode sinalizar:

- **Dificuldade de utilização do crédito**: o consorciado pode não ter encontrado o bem desejado ou enfrentar entraves burocráticos;
- **Atualização do valor do crédito**: em grupos mais antigos, o crédito contemplado pode estar defasado em relação ao valor atual do bem, exigindo reajuste ou complemento;
- **Risco de inadimplência futura**: consorciados que não conseguem utilizar o crédito podem parar de pagar as parcelas restantes, elevando a inadimplência do grupo.

Recomenda-se que as administradoras monitorem esses grupos de perto, acompanhando ativamente os consorciados contemplados para garantir a utilização do crédito e mitigar riscos operacionais e financeiros.

**Consulta 8 - Risco Combinado**

Pergunta de Negócio

>Quais grupos apresentam simultaneamente inadimplência elevada e exclusões relevantes?

Objetivo de criar um ranking de grupos potencialmente mais críticos.

In [0]:
%sql
SELECT
nome_da_administradora,
codigo_do_grupo,
ROUND(taxa_inadimplencia * 100, 2) AS inadimplencia_percentual, quantidade_de_cotas_excluidas, quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao AS credito_pendente, faixa_inadimplencia
FROM workspace.mvp_consorcios.gold_grupos_risco
WHERE faixa_inadimplencia IN ('Alto', 'Muito alto')
AND quantidade_de_cotas_excluidas > 0
ORDER BY
taxa_inadimplencia DESC,
quantidade_de_cotas_excluidas DESC
LIMIT 30;

nome_da_administradora,codigo_do_grupo,inadimplencia_percentual,quantidade_de_cotas_excluidas,credito_pendente,faixa_inadimplencia
KSK ADM CONS LTDA.,00707,87.86,2,4,Muito alto
KSK ADM CONS LTDA.,00706,87.14,1110,43,Muito alto
KSK ADM CONS LTDA.,00703,74.18,3575,26,Muito alto
KSK ADM CONS LTDA.,00702,69.56,4553,17,Muito alto
KSK ADM CONS LTDA.,00704,67.7,5606,57,Muito alto
ALPHA ADM CONS LTDA,1004,59.76,4448,31,Muito alto
PORTO SEGURO ADM. CONS. LTDA,I0387,56.56,3,21,Muito alto
ALPHA ADM CONS LTDA,S1005,52.11,3902,63,Muito alto
KSK ADM CONS LTDA.,00701,50.86,5766,37,Muito alto
MULTIMARCAS ADM.CONS.LTDA.,00990,50.0,3229,5,Muito alto


Os grupos que aparecem no topo da lista de consulta combinam diversos sinais de atenção operacional.

A Consulta 8 combina dois indicadores de risco — inadimplência e exclusões de cotas — para identificar os grupos mais críticos da operação. Ao filtrar apenas grupos cuja faixa de inadimplência é classificada como **Alto** ou **Muito alto** **e** que possuem ao menos uma cota excluída, a consulta isola grupos que acumulam dois problemas simultaneamente.

**O que os resultados revelam**

- **Inadimplência elevada**: o percentual de cotas inadimplentes (`inadimplencia_percentual`) indica a proporção de consorciados que não estão honrando o pagamento das parcelas. Grupos no topo do ranking apresentam taxas expressivas, sinalizando deterioração do comportamento de pagamento.

- **Exclusões de cotas**: a coluna `quantidade_de_cotas_excluidas` mostra quantas cotas foram retiradas do grupo, geralmente por cancelamento, inadimplência persistente ou outros motivos operacionais. Um número elevado de exclusões em conjunto com alta inadimplência indica que o grupo está perdendo participantes ativos, o que pode comprometer a sustentabilidade do consórcio.

- **Crédito pendente de utilização**: a coluna `credito_pendente` traz um terceiro sinal de alerta — consorciados contemplados que ainda não utilizaram o crédito. Quando somado à inadimplência e às exclusões, esse indicador reforça o cenário de risco, pois cotas pendentes associadas a grupos problemáticos podem gerar inadimplência futura.

**Por que esse cruzamento é relevante**

Cada indicador isolado já sinaliza problemas, mas **a combinação de inadimplência alta com exclusões relevantes** aponta grupos que enfrentam múltiplas frentes de risco ao mesmo tempo. Esses grupos:

- Têm menor capacidade de absorção de perdas, pois perdem cotas ativas que contribuem financeiramente para o grupo;
- Apresentam risco de **contágio**, em que a saída de cotas e a inadimplência podem desestimular os demais consorciados adimplentes;
- Demandam **ação prioritária de gestão**, incluindo renegociação de débitos, cobrança ativa e revisão das condições de contemplação.

**Recomendações**

1. **Monitoramento intensivo**: os grupos no topo do ranking devem receber acompanhamento próximo e frequente por parte da administradora.
2. **Plano de recuperação**: elaborar estratégias de cobrança e renegociação específicas para esses grupos, visando reduzir a inadimplência e conter novas exclusões.
3. **Revisão de critérios de concessão**: investigar se os grupos críticos compartilham características comuns (perfil dos consorciados, prazo, tipo de bem) que possam informar ajustes na política de crédito.
4. **Comunicação com consorciados**: contatar ativamente os consorciados adimplentes desses grupos para preservar a confiança e evitar um efeito cascata de cancelamentos.

Em síntese, a consulta de risco combinado funciona como um **triagem de prioridade máxima**: identifica os grupos que concentram os maiores problemas operacionais e financeiros, orientando ações de mitigação de forma focada e eficiente. Mas ao mesmo tempo não permite concluir as cauasa da inadimplência ou das exclusões, pois não há histórico de pagamentos nem dados individuais dos consorciados.


Consulta 9 - Distribuição dos Grupos por Faixa de Risco

Pergunta de Negócio

>Como os grupos imobiliários estão distribuídos entre os diferentes níveis de risco de inadimplência?

Objetivo de investigar a concentração dos grupos nas faixas de risco.

In [0]:
%sql
SELECT
faixa_inadimplencia,
COUNT(*) AS quantidade_grupos
FROM workspace.mvp_consorcios.gold_grupos_risco
GROUP BY faixa_inadimplencia
ORDER BY
CASE faixa_inadimplencia
WHEN 'Baixo' THEN 1
WHEN 'Moderado' THEN 2
WHEN 'Alto' THEN 3
WHEN 'Muito alto' THEN 4
ELSE 5
END;

faixa_inadimplencia,quantidade_grupos
Baixo,855
Moderado,1324
Alto,554
Muito alto,102


Databricks visualization. Run in Databricks to view.

1. **Concentração no centro da distribuição**: A maioria dos grupos (1.324, equivalente a ~46,8% do total) está classificada na faixa **Moderado**. Isso indica que a maior parte dos grupos apresenta um nível intermediário de inadimplência — nem crítico, nem livre de risco. É um sinal de que muitos grupos merecem acompanhamento, embora não demandem ação imediata.

2. **Grupos de baixo risco**: 855 grupos (~30,2%) estão na faixa **Baixo**. Esses grupos apresentam taxas de inadimplência reduzidas e representam o segmento mais saudável da operação. Ainda assim, é recomendável manter monitoramento periódico para identificar precocemente qualquer deterioração.

3. **Grupos de alto risco**: 554 grupos (~19,5%) estão na faixa **Alto**. Esse volume é relevante — quase um em cada cinco grupos — e indica que existe uma porção significativa da carteira com sinais claros de pressão operacional. Esses grupos devem ser priorizados para ações de gestão, como cobrança ativa e renegociação de débitos.

4. **Grupos de risco muito alto**: 102 grupos (~3,6%) estão na faixa **Muito alto**. Embora sejam numericamente menores, esses grupos concentram o maior risco individual. Cada grupo nessa faixa exige atenção prioritária e individualizada, pois a taxa de inadimplência atingiu níveis que comprometem a sustentabilidade do consórcio.

A distribuição apresenta um formato aproximado de sino assimétrico, com concentração na faixa Moderado e caudas que se estendem para os extremos. O fato de **656 grupos (23,1% do total)** estarem classificados nas faixas Alto e Muito alto combinados indica que **quase um em cada quatro grupos** apresenta sinais de atenção significativos. Esse percentual não é negligenciável e sugere que a inadimplência é um fenômeno relevante na carteira de consórcios imobiliários analisada.

Por outro lado, a maior parte dos grupos (76,9%) está em faixas de menor risco (Baixo e Moderado), o que indica que a operação geralmente está sob controle, mas com bolsões de risco que demandam intervenção focalizada.

> **Nota**: Esta análise reflete apenas o recorte da data-base de julho de 2026. Não é possível inferir tendência ou evolução temporal, pois não há dados históricos disponíveis.

###Criação de visão SQL - Grupos com maior taxa de inadimplência da base

Criaremos uma camada de apresentação chamada vw_monitoramento_risco, baseada na tabela Gold gold_grupos_risco, filtrando apenas grupos com cotas ativas e facilitando o consumo de dados por relatórios e usuários finais.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.mvp_consorcios.vw_monitoramento_risco AS SELECT
nome_da_administradora,
cnpj_da_administradora,
codigo_do_grupo,
data_base,
prazo_do_grupo_em_meses,
valor_medio_do_bem,
taxa_de_administracao,
quantidade_cotas_ativas_calculada,
quantidade_cotas_inadimplentes,
taxa_inadimplencia,
quantidade_de_cotas_excluidas,
quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao,
faixa_inadimplencia,
faixa_prazo
FROM workspace.mvp_consorcios.gold_grupos_risco
WHERE quantidade_cotas_ativas_calculada > 0;

Abaixo apresentaremo a execução da view criada acima para a consulta que responde algo como: "Quais são os 20 grupos com maior taxa de inadimplência da base?"

In [0]:
%sql
SELECT *
FROM workspace.mvp_consorcios.vw_monitoramento_risco
ORDER BY taxa_inadimplencia DESC
LIMIT 20;

nome_da_administradora,cnpj_da_administradora,codigo_do_grupo,data_base,prazo_do_grupo_em_meses,valor_medio_do_bem,taxa_de_administracao,quantidade_cotas_ativas_calculada,quantidade_cotas_inadimplentes,taxa_inadimplencia,quantidade_de_cotas_excluidas,quantidade_de_cotas_ativas_com_credito_pendente_de_utilizacao,faixa_inadimplencia,faixa_prazo
KSK ADM CONS LTDA.,62798475,00707,2026-07-01,150,195143.77,18.5559,313,275,0.8785942492012779,2,4,Muito alto,121 a 180 meses
KSK ADM CONS LTDA.,62798475,00706,2026-07-01,220,382892.29,22.9053,4308,3754,0.8714020427112349,1110,43,Muito alto,181 a 240 meses
KSK ADM CONS LTDA.,62798475,00703,2026-07-01,220,264606.48,23.2181,972,721,0.7417695473251029,3575,26,Muito alto,181 a 240 meses
KSK ADM CONS LTDA.,62798475,00702,2026-07-01,220,281639.71,23.6552,841,585,0.6956004756242569,4553,17,Muito alto,181 a 240 meses
KSK ADM CONS LTDA.,62798475,00704,2026-07-01,220,340544.2,23.4816,1935,1310,0.6770025839793282,5606,57,Muito alto,181 a 240 meses
ALPHA ADM CONS LTDA,90982679,1004,2026-07-01,240,456167.0,24.0441,1178,704,0.597623089983022,4448,31,Muito alto,181 a 240 meses
PORTO SEGURO ADM. CONS. LTDA,48041735,I0387,2026-07-01,200,73537.26,23.0,900,509,0.5655555555555556,3,21,Muito alto,181 a 240 meses
ALPHA ADM CONS LTDA,90982679,S1005,2026-07-01,300,208081.0,23.5369,4235,2207,0.521133412042503,3902,63,Muito alto,Acima de 240 meses
KSK ADM CONS LTDA.,62798475,00701,2026-07-01,220,192054.03,23.0602,930,473,0.5086021505376344,5766,37,Muito alto,181 a 240 meses
MULTIMARCAS ADM.CONS.LTDA.,04124922,00990,2026-07-01,156,77316.47,27.4615,26,13,0.5,3229,5,Muito alto,121 a 180 meses


###Estatísticas Descritivas


In [0]:
%sql
SELECT
COUNT(*) AS quantidade_grupos,
COUNT(DISTINCT cnpj_da_administradora) AS quantidade_administradoras,
MIN(valor_medio_do_bem) AS menor_valor_medio_bem,
AVG(valor_medio_do_bem) AS media_valor_medio_bem,
MAX(valor_medio_do_bem) AS maior_valor_medio_bem,
MIN(taxa_de_administracao) AS menor_taxa_administracao,
AVG(taxa_de_administracao) AS media_taxa_administracao,
MAX(taxa_de_administracao) AS maior_taxa_administracao
FROM workspace.mvp_consorcios.gold_grupos_risco;

quantidade_grupos,quantidade_administradoras,menor_valor_medio_bem,media_valor_medio_bem,maior_valor_medio_bem,menor_taxa_administracao,media_taxa_administracao,maior_taxa_administracao
2835,69,113.68,253236.28610582012,1598745.75,0.0,19.305103530864194,35.0



A consulta acima executa uma análise estatística descritiva sobre a tabela `gold_grupos_risco`, com o objetivo de fornecer uma visão geral e consolidada da base de dados. Essa etapa é fundamental para entender a distribuição e a amplitude dos dados antes de aprofundar análises mais específicas.

**O que cada métrica representa:**

- **`quantidade_grupos`** : total de registros (grupos de consórcio) presentes na tabela. Indica o volume total da base analisada.
- **`quantidade_administradoras`** : número de administradoras distintas, identificadas pelo CNPJ. Mostra a diversidade de entidades operando na base.
- **`menor_valor_medio_bem`**, **`media_valor_medio_bem`** e **`maior_valor_medio_bem`**: respectivamente o valor mínimo, a média e o valor máximo do `valor_medio_do_bem`. Esses indicadores revelam a faixa de valores dos bens contemplados nos grupos, permitindo identificar desde grupos com bens de baixo valor até grupos com bens de alto valor.
- **`menor_taxa_administracao`**, **`media_taxa_administracao`** e **`maior_taxa_administracao`**: mínimo, média e máximo da `taxa_de_administracao`. Mostram a amplitude das taxas cobradas pelas administradoras, o que pode influenciar a competitividade e o custo do consórcio para o participante.

Com base nessa consulta, é possível extrair as seguintes observações:

1. **Abrangência da base**: o total de grupos e de administradoras distintas confirma que a análise cobre um conjunto amplo e representativo do mercado de consórcios, conferindo robustez estatística às conclusões das consultas anteriores.

2. **Dispersão do valor médio do bem**: a diferença entre o menor e o maior valor médio do bem evidencia a heterogeneidade dos grupos — existem grupos voltados a bens de menor valor (ex.: motocicletas, eletrodomésticos) e grupos para bens de alto valor (ex.: imóveis, caminhões). A média fornece um ponto central, mas a amplitude indica que a base é bastante diversificada.

3. **Amplitude da taxa de administração**: a variação entre a menor e a maior taxa de administração mostra que as administradoras adotam estratégias de precificação distintas. Taxas mais altas podem estar associadas a serviços adicionais ou a perfis de risco mais elevados, enquanto taxas mais baixas podem refletir maior escala operacional ou menor agregação de serviços.

4. **Importância para análises subsequentes**: essas estatísticas servem como referência para contextualizar os resultados das consultas anteriores. Por exemplo, ao analisar a inadimplência por administradora (Consultas 4 e 6), é relevante saber a faixa de taxas de administração e o valor dos bens envolvidos, pois esses fatores podem influenciar o comportamento de pagamento dos consorciados.

Em síntese, essa etapa de estatísticas descritivas funciona como um **panorama da base de dados**, fornecendo parâmetros de comparação e permitindo validar se a amostra é representativa para as análises de risco e inadimplência desenvolvidas nas seções anteriores.


###Correlação Exploratória

Esta etapa visa realizar uma análise estatística exploratória para verificar se existe alguma relação entre atributos, por exemplo a **inadimplência** dos grupos de consórcio e duas características que podem ter influência ou estare relacionadas:

_1- Taxa de administração;_

_2- Prazo do grupo (em meses)._

No final, gostaríamos de responder perguntas como "Grupos com taxas de administração maiores tendem a apresentar mais inadimplência?" ou ainda, "Grupos com prazo mais longo possume memos inadimplência mais elevada?"

In [0]:
correlacao_taxa_adm = (                                                             # Correlação entre taxa de administração e inadimplência
    df_grupos_risco
    .filter(
        F.col("taxa_de_administracao").isNotNull()                                  # Filtra valores nulos
        & F.col("taxa_inadimplencia").isNotNull()
    )
    .stat.corr("taxa_de_administracao", "taxa_inadimplencia")                       # Calcula correlação de Pearson
)
correlacao_prazo = (                                                                # Correlação entre prazo do grupo e inadimplência
    df_grupos_risco
    .filter(
        F.col("prazo_do_grupo_em_meses").isNotNull()                                # Filtra valores nulos
        & F.col("taxa_inadimplencia").isNotNull()
    )
    .stat.corr("prazo_do_grupo_em_meses", "taxa_inadimplencia")                     # Calcula correlação de Pearson
)
print("Correlação taxa de administração x inadimplência:", correlacao_taxa_adm)     # Exibe resultado
print("Correlação prazo x inadimplência:", correlacao_prazo)                        # Exibe resultado

Correlação taxa de administração x inadimplência: 0.31376468504161004
Correlação prazo x inadimplência: 0.18977005289631171


**Interpretação das correlações**

A correlação mede a associação linear entre duas variáveis, variando de -1 a 1.

- Valor próximo de 1: associação linear positiva.
- Valor próximo de -1: associação linear negativa.
- Valor próximo de 0: associação linear fraca ou inexistente.

**Resultados observados**

_**Taxa de administração x inadimplência**_

Correlação = **0,314**

Foi observada uma correlação positiva moderada-fraca entre a taxa de administração e a taxa de inadimplência. Isso sugere que grupos com taxas de administração mais elevadas tendem, em média, a apresentar níveis maiores de inadimplência.
Entretanto, o coeficiente encontrado (0,314) está longe de uma correlação forte, indicando que a taxa de administração explica apenas uma parcela limitada da variabilidade da inadimplência. Outros fatores não presentes nesta análise provavelmente possuem influência mais significativa sobre o comportamento dos grupos.

_**Prazo do grupo x inadimplência**_

Correlação = **0,190**

De maneira análoga, foi observada uma correlação positiva fraca entre o prazo do grupo e a taxa de inadimplência. O resultado sugere uma leve tendência de grupos com prazos mais longos apresentarem inadimplência um pouco maior.

No entanto, o valor de 0,190 é relativamente baixo, indicando que o prazo do grupo, isoladamente, possui pouca capacidade de explicar as diferenças de inadimplência observadas entre os grupos analisados.

**Comentários**

>Entre as variáveis analisadas, a taxa de administração apresentou uma associação mais relevante com a inadimplência do que o prazo do grupo. Ainda assim, ambas as correlações podem ser consideradas de baixa magnitude para fins preditivos.

>Esses resultados sugerem que a inadimplência é um fenômeno multifatorial, influenciado por diversos aspectos além daqueles disponíveis na base analisada, tais como perfil dos consorciados, condições econômicas, segmentação de mercado, concentração geográfica, valor do crédito contratado e características específicas de cada administradora.

>Por fim, é importante ressaltar que correlação não implica causalidade. Os resultados indicam associação estatística entre as variáveis, mas não permitem concluir que uma variável seja a causa direta da outra.

##Segurança, Ética e Privacidade

Os arquivos utilizados contêm informações sobre administradoras e grupos de consórcios. Não foram identificados dados pessoais de consorciados, como nome, CPF, endereço ou renda.

Ainda assim, foram adotados os seguintes princípios:

1. Utilização dos dados somente para finalidade acadêmica declarada.
2. Preservação da origem dos arquivos e do dicionário de dados.
3. Não inferência de comportamento de indivíduos.
4. Separação entre dados brutos, tratados e analíticos.
5. Registro das transformações aplicadas.
6. Não divulgação de credenciais, tokens ou informações de acesso ao ambiente.
7. Interpretação dos indicadores como sinais exploratórios, e não como prova de irregularidade.
8. Não utilização dos rankings para acusação ou avaliação definitiva das administradoras.

Caso dados pessoais sejam incorporados futuramente, será necessário avaliar finalidade, base legal, minimização, controle de acesso, anonimização e retenção, em conformidade com a LGPD.

## Metadados e linhagem

**Origem**
- Arquivo consolidado de segmentos, data-base julho de 2026. (Não utilizado nesse MVP)
- Arquivo de grupos de bens imóveis, data-base julho de 2026.
- Planilha de significado dos campos e das métricas.

**Granularidade**
- Segmentos: uma linha por administradora, data-base e código de segmento. (Não utilizado nesse MVP)
- Grupos imobiliários: uma linha por administradora, data-base, código do grupo e código do segmento.

**Transformações realizadas**
1. Leitura dos CSVs com separador ponto e vírgula.
2. Preservação inicial de todas as colunas como texto.
3. Remoção de espaços adicionais.
4. Padronização dos nomes das colunas.
5. Preservação do CNPJ como texto de oito caracteres.
6. Conversão da data-base AAAAMM em data.
7. Conversão da vírgula decimal para ponto.
8. Conversão dos tipos numéricos.
9. Cálculo das cotas ativas e inadimplentes.
10. Cálculo da taxa de inadimplência.
11. Agregação por administradora.
12. Comparação com o arquivo consolidado.
13. Construção de tabelas Bronze, Silver e Gold.
14. Construção de visão SQL para monitoramento.

**Linhagem resumida**
Arquivos CSV e XLSX
→ tabelas Bronze
→ limpeza e validação
→ tabelas Silver
→ indicadores e agregações
→ tabelas Gold
→ consultas, visões e gráficos.

## Autoavaliação

Esta seção apresenta a autoavaliação do trabalho, contemplando a discussão sobre o atingimento dos objetivos, as dificuldades encontradas durante a execução e os trabalhos futuros para enriquecer a solução.

### Objetivos atingidos

- ✅ Construir uma solução mínima viável de dados para organizar, tratar, armazenar e analisar informações públicas sobre grupos de consórcios imobiliários
- ✅ Criar indicadores proporcionais que permitam comparar grupos de tamanhos diferentes
- ✅ Identificar administradoras e grupos com maior proporção de cotas inadimplentes, excluídas e créditos pendentes
- ✅ Persistir os dados em camadas (Bronze, Silver, Gold) no Databricks
- ✅ Responder às 10 perguntas de negócio formuladas
- ✅ Avaliar a qualidade dos dados em múltiplas dimensões

### Dificuldades encontradas

1. **Tratamento do encoding e separador decimal**: O arquivo CSV utiliza `windows-1252` e vírgula como separador decimal, exigindo leitura manual e conversão controlada para evitar perda de dados.
2. **Preservação do CNPJ**: A leitura automática do Spark converteria o CNPJ para número, perdendo zeros à esquerda. Foi necessário ler todas as colunas como texto e aplicar `lpad` manualmente.
3. **Definição da chave de negócio**: A identificação da granularidade correta exigiu testes de unicidade, pois a inclusão ou exclusão do código do segmento poderia alterar a chave.
4. **Ausência de dados históricos**: A análise ficou limitada a uma única data-base, impossibilitando a verificação de tendências ou evolução temporal da inadimplência.
5. **Classificação de risco**: A criação de faixas de inadimplência (Baixo, Moderado, Alto, Muito alto) não se baseou em normas do BCB, sendo uma decisão acadêmica que exige validação com especialistas do domínio.
6. **Correlações de baixa magnitude**: As correlações encontradas entre taxa de administração/prazo e inadimplência foram fracas, evidenciando a natureza multifatorial do fenômeno e a limitação dos dados disponíveis.
7. **Ambiente Databricks Free Edition**: Limitações de compute e armazenamento no ambiente gratuito restringiram a escala de testes e a possibilidade de experimentar pipelines mais complexos.

### Limitações

1. A análise utiliza somente a data-base de julho de 2026.
2. Não há histórico mensal para estudar tendências.
3. Não há dados geográficos.
4. Não há dados individuais dos consorciados.
5. Não há valor individual da parcela ou saldo devedor.
6. Não há motivo de exclusão ou atraso.
7. Não há identificação das contemplações por sorteio ou lance.
8. A classificação de risco foi criada exclusivamente para fins acadêmicos.
9. Os indicadores representam associação descritiva e não causalidade.
10. O valor médio do bem não deve ser tratado como saldo devedor ou valor inadimplente.

### Trabalhos Futuros

1. Incorporar arquivos mensais anteriores e posteriores a julho de 2026.
2. Construir série temporal da inadimplência.
3. Identificar entrada e encerramento de grupos ao longo do tempo.
4. Adicionar descrição oficial dos códigos de índice de correção.
5. Integrar dados econômicos, como inflação, juros e desemprego.
6. Incorporar localização da sede ou área de atuação das administradoras.
7. Criar controles automatizados de qualidade para cada nova carga.
8. Desenvolver alertas para variações anormais de inadimplência.
9. Avaliar modelos estatísticos ou de aprendizado de máquina apenas após obter histórico suficiente.
10. Investigar formalmente as diferenças entre os grupos e os valores consolidados.
11. Implementar carga incremental e controle de versões.
12. Criar painel executivo com filtros de administradora, prazo e faixa de risco.
13. Incluir os dados provenientes do arquivo de dados do segmento (não utilizado nesse MVP)

### Conclusão

O MVP demonstrou a construção de uma solução de dados para análise de grupos de consórcios imobiliários no Databricks Free Edition.

Os arquivos originais foram ingeridos em uma camada bruta, padronizados e convertidos para tipos adequados em uma camada tratada e transformados em indicadores na camada analítica. Foram avaliadas completude, unicidade, validade e consistência dos dados.

A principal métrica desenvolvida foi a taxa de inadimplência, calculada como a proporção entre as cotas ativas inadimplentes e o total de cotas ativas consideradas. O indicador foi agregado de forma ponderada, evitando que grupos de tamanhos diferentes recebessem o mesmo peso.

A solução permitiu comparar administradoras, identificar grupos com maiores sinais de atenção, avaliar diferenças entre faixas de prazo e confrontar os resultados calculados com o arquivo consolidado.

As conclusões devem ser interpretadas como descritivas. Os dados disponíveis não permitem determinar causalidade, calcular o valor financeiro exato inadimplente ou explicar os motivos individuais dos atrasos. A ausência de histórico também impede a análise de tendência.

Como evolução, recomenda-se incorporar diversas datas-base, variáveis econômicas, informações geográficas e dados adicionais sobre os grupos. Com isso, a solução poderá evoluir de um MVP descritivo para uma plataforma de monitoramento temporal e apoio à decisão.

## Referências utilizadas

Material Didático da Especialização PUC-RIO - Engenharia de Dados
- Fundamentos, modelo relacional, restrições de integridade e SQL básico.
- Projeto conceitual e lógico, SQL DDL e SQL DML.
- Consultas, agregações, junções, visões e projeto físico.
- Modelagem, projeto e implementação de data warehouses.
- Qualidade, metadados e linhagem de dados.
- Data lakes, ingestão e ETL.
- Segurança, ética e LGPD.

Dicionário de campos e métricas dos dados de consórcios. Fonte: https://www.bcb.gov.br/estabilidadefinanceira/consorciobd

Arquivos de segmentos consolidados e grupos de bens imóveis, data-base julho de 2026. Fonte: https://www.bcb.gov.br/estabilidadefinanceira/consorciobd